# Anima: the framed nano arms read each description in their own frame and set the three group sliders

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/anima-trainer/blob/main/notebooks/anima_colab_framed_readers.ipynb)

The follow-up to the group readers: the same descriptions, tiers and pictures, but the small byte-level nano arms trained on
framed records read each description written in that frame grammar (one 32-byte record frame per attribute, then the question
for one attribute and its value), and each slider of the group is set from its own fetch. Qwen3's plain-description readings
are the reference; the picked arm's body before any framed training and its untrained start read in the same frame beside it.
The readings were made beforehand and live in the data repo's `beatrix/cgf_readers.json`.

**Nothing is drawn twice.** The plain prompt's pictures and the group at amount 1 come from
[AbstractPhil/geolip-beatrix-anima-data](https://huggingface.co/datasets/AbstractPhil/geolip-beatrix-anima-data); a picture
identical to one already drawn is copied; each reader's sets are saved there under `sets/cgf2_*` as soon as they are drawn.

**Content screen.** As before: a picture rated questionable or explicit, or read as child, loli or aged down above 0.2, is
left off the sheets and its set is not uploaded; the fourth character's pictures carry "child, loli, aged down" in the
negative prompt.

**The rule (fixed before the run).** For each reader, tier, pair and side, the 16 descriptions' pictures each against their
cell's plain picture: carried when at least 12 of 16 move the side's way, beyond 3 standard errors; a reader drives the group
when all three pairs are carried on both sides; the untrained control must drive none.

**Cost.** At most 320 pictures (Qwen3's 64 reused when the earlier sets are on the runtime) at 768 px and 30 steps, up to five
predictions a step; about an hour on the RTX PRO 6000. Results: `experiments/e055_anima_framed_group_readers` of
[AbstractPhil/geolip-beatrix-anima](https://huggingface.co/AbstractPhil/geolip-beatrix-anima).


## 0 · Confirm the GPU and disk

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1

## 1 · Bootstrap: the runner at its release, and the dependencies (restart ONCE the first time)

Clones the runner at the release this notebook was written for, and the diffusion-pipe fork beside it (its Anima model
code draws the pictures), then installs the dependencies. **Run the cell; if the kernel restarts, run it again** (it
then skips the install).

In [ ]:
import os, sys, subprocess
TAG = "v0.7.0"                       # this notebook's release of the runner
REPO = os.environ.get("ANIMA_REPO", "/content/anima-trainer")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/AbstractEyes/anima-trainer.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--quiet", "--tags", "origin"], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "--quiet", TAG], check=True)
sys.path.insert(0, REPO)
import anima_colab
if anima_colab.install(REPO, dp_url=anima_colab.DP_FORK_URL, similarity=False):   # + the diffusion-pipe fork
    os.kill(os.getpid(), 9)             # -> restart; reconnect and RE-RUN this cell (then it skips)

## 2 · Set up

Checks the GPU, finds the fork and downloads the three model files. Needs a **write** `HF_TOKEN` in Colab Secrets: the
results and the drawn sets are uploaded.

In [ ]:
from geolip_anima_trainer.anima_runner import AnimaRunner
a = AnimaRunner()      # Anima-Base v1.0 at 768 px; results -> AbstractPhil/geolip-beatrix-anima, sets -> ...-anima-data
a.setup()              # env + HF login + GPU check + the fork + the three model files (~5.6 GB)

## 3 · The readers on the group

Downloads the readers file, prints the job's size (which sets come from the data repo, how many distinct pictures it
draws), then time spent and time left with one line per set, and at the end one line per tier and reader: the verdict,
each pair's share of descriptions moved its way on each side, and how many pictures moved all three. A run already done
is skipped (`force=True` reruns the reads; the sets come back from the data repo).


In [ ]:
from huggingface_hub import hf_hub_download
readers = hf_hub_download("AbstractPhil/geolip-beatrix-anima-data", "beatrix/cgf_readers.json", repo_type="dataset")
meta = a.run_group_readers(readers, test="e055")     # the framed arms: one fetch per slider; 5 readers x 2 tiers x 32


## Notes

- **Where things land:** `experiments/e055_anima_framed_group_readers/` in the experiments repo (README with the question, the
  design, the rule fixed before the run and the result; `meta.json`; `readers.json`; `result.json` with every picture's
  score on every pair; `tagger_probs.npz`; `sheet_trained.jpg` and `sheet_unseen.jpg`). The sets:
  `sets/cgf2_<reader>_<tier>-<key>/` in the data repo. Locally: `/content/anima_data/`.
- **After a disconnect:** run steps 1-3 again. Finished sets come back from the data repo; only the unfinished set is
  drawn again.
- **The pictures** are outputs of Anima-Base v1.0 (CircleStone Labs Non-Commercial License: outputs are not derivatives
  of the model; the restriction applies to the model and its derivatives, such as LoRAs).
